# AIFM PE Buyout Fund

This notebook presents a private equity risk-monitoring workflow for a simulated closed-ended AIF. The fund is a 2018-vintage European mid-market buyout strategy with portfolio companies across technology, healthcare, industrials, consumer, and energy-transition sectors.

The analysis focuses on private-capital indicators: portfolio company overview, independent appraisals and covenant headroom, IRR and DPI/RVPI/TVPI multiples, the J-curve, exit waterfalls, fund cash management, the value bridge, unfunded commitments and funding liquidity, the PME benchmark comparison, NAV stress scenarios, and sustainability indicators. PE data lives in dedicated database tables (companies, cash flows, quarterly appraisals) — not in the shared daily position snapshot.

> **Output gallery:** All tables and plots generated by this notebook are saved in the [fig/AIFM_PE_Buyout](../../fig/AIFM_PE_Buyout) folder. Readers who prefer to review the generated outputs directly can browse that folder without running the notebook.

In [ ]:
import warnings

from fund_risk_workflow.data.setup_db import run as setup_db
from fund_risk_workflow.data.mock_bloomberg import MockBloomberg as Bloomberg

import fund_risk_workflow.data.database as db
import fund_risk_workflow.risk.esg_utils as esg_u
import fund_risk_workflow.ui.print_html_utils as phtml
import fund_risk_workflow.ui.pe_buyout_display as ped

warnings.filterwarnings("ignore")

setup_db()
ENGINE = db.get_engine()
BBG = Bloomberg()

## 1. Fund Setup and Risk Policy

### 1.1 Fund Example

The fund profile below sets the operating context for the risk workflow. It defines the strategy, fund type, base currency, reporting setup, and monitoring framework used by the calculations that follow.

In [ ]:
# Display fund overview banner — fund identity and risk methodology framework
FUND_ID = 'AIFM_PE_Buyout'
phtml.display_fund_overview_banner(
    fund_id=FUND_ID,
    engine=ENGINE,
    export_id="01",
)

> Note: Fund characteristics, risk limits, methodologies, and reporting parameters are simulated. They are used to show how a fund-level risk framework can be represented in a structured workflow.

---

### 1.2 Risk Management Policy Parameters

The fund's risk parameters are sourced from the Risk Management Policy configuration. The PE stress scenario magnitudes and the capital-call stress assumption are documented in the risk policy rather than in notebook code.

In [ ]:
# Display Risk Management Policy parameters from fund reference data
phtml.display_fund_rmp_parameters(
    fund_id=FUND_ID,
    engine=ENGINE,
    export_id="02",
)

### 1.3 Implementation Context

The analysis is performed as of a fixed valuation date; appraisals and ESG indicators use the matching reporting quarter.

In [ ]:
# Fixed valuation date and reporting quarter for all computations
from fund_risk_workflow.config import QUARTER, VALUATION_DATE
VALUATION_DATE, QUARTER

The workflow builder reads the populated PE tables (`pe_funds`, `pe_portfolio_companies`, `pe_fund_investments`, `pe_cash_flows`, `pe_nav_history`, `pe_valuation_report`, `pe_fund_cash_management`) and computes every result used in this notebook. From this point onward, code cells contain only display calls.

In [ ]:
# Build the full PE buyout monitoring result set
from fund_risk_workflow.pipeline.pe_buyout_workflow import build_pe_buyout_workflow

workflow = build_pe_buyout_workflow(
    engine=ENGINE,
    bbg=BBG,
    fund_id=FUND_ID,
    valuation_date=VALUATION_DATE,
    quarter=QUARTER,
)

---

## 2. Portfolio Company Overview

One row per investment: sector, country, stage, status, investment date, cost basis, ownership, and entry multiples. For realised investments, the exit multiple is derived from the latest independent appraisal at the exit date over cost basis.

In [ ]:
ped.display_portfolio_overview(workflow["portfolio_overview"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="03")

---

## 3. Valuation and Covenant Monitoring

Portfolio companies are valued quarterly using independent appraisal inputs. The monitor shows the latest appraised NAV, LTM EBITDA, EV/EBITDA, leverage against covenant, and covenant headroom. Valuation risk is treated as model and appraisal risk rather than daily market-price risk.

> A leverage ratio flagged as not meaningful (negative EBITDA) is shown as missing; headroom below 20% renders in red.

In [ ]:
ped.display_valuation_monitor(workflow["valuation_monitor"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="04")

---

## 4. Performance and J-Curve

Fund performance uses the standard private-capital measures: XIRR on LP cash flows plus terminal NAV, and DPI / RVPI / TVPI against paid-in capital. The J-curve tracks quarterly capital calls, fees, distributions, cumulative net cash flow, and the multiple evolution over the fund life.

In [ ]:
ped.display_performance_summary(workflow["performance"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="05")

In [ ]:
ped.display_multiples_by_company(workflow["performance"]["multiples_by_company"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="06")

In [ ]:
ped.plot_j_curve(workflow["j_curve"], FUND_ID, valuation_date=VALUATION_DATE, export_id="07")

---

## 5. Exit Waterfall

The exit waterfall shows how proceeds from each realised company sale are allocated between LPs and the GP. The simulated waterfall follows a European-style sequence: return of capital, preferred return at the hurdle rate, GP catch-up, and the carried-interest split.

In [ ]:
ped.plot_exit_waterfalls(workflow["exit_waterfalls"], FUND_ID, valuation_date=VALUATION_DATE, export_id="08")

---

## 6. Fund Cash Management

Fund-level cash, subscription credit facility usage, net cash position, and cumulative interest earned versus paid. The subscription line is short-term bridge financing backed by unfunded LP commitments.

In [ ]:
ped.plot_cash_management(workflow["cash_summary"], FUND_ID, valuation_date=VALUATION_DATE, export_id="09")

---

## 7. Return Attribution — Value Bridge

The value bridge decomposes equity value creation into EBITDA growth, multiple expansion, deleveraging, and distributions, separating operational value creation from valuation and capital-structure effects. Reconciliation gaps against appraised values are shown, not suppressed: for unrealised companies the appraiser NAV can include DCF or growth premia outside the EV/EBITDA bridge.

In [ ]:
ped.plot_value_bridge_by_company(workflow["value_bridge"], FUND_ID, valuation_date=VALUATION_DATE, export_id="10")

In [ ]:
ped.plot_value_bridge_fund(workflow["value_bridge"], FUND_ID, valuation_date=VALUATION_DATE, export_id="11")

In [ ]:
ped.display_bridge_gaps(workflow["value_bridge"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="12")

---

## 8. Commitments and Funding Liquidity

For a closed-ended PE fund, liquidity risk sits on the liability side: unfunded commitments, fees, and subscription-line obligations. The coverage ratio compares cash, sub-line headroom, and trailing-12-month distributions against trailing-12-month calls and fees. The capital-call stress assumes an accelerated drawdown of the documented share of unfunded commitments.

The closed-ended liquidity bucket table confirms the asset side sits in the > 1 year bucket by construction.

In [ ]:
ped.display_commitment_liquidity(workflow["commitment_liquidity"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="13")

In [ ]:
ped.display_liquidity_buckets(workflow["commitment_liquidity"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="14")

---

## 9. PME Benchmark — Long-Nickels

The Public Market Equivalent answers whether LPs would have done better in public markets: each capital call is replicated as an index purchase and each distribution as an index sale, with the residual index portfolio as the PME terminal NAV. Alpha = PE IRR − PME IRR.

Benchmark: Euro Stoxx 50 (SX5E), EUR-denominated and appropriate for a European buyout fund. The series comes from the local market-data cache, requested from the first actual fund cash-flow date.

In [ ]:
ped.display_pme_summary(workflow["pme"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="15")

In [ ]:
ped.plot_pme(workflow["pme"], FUND_ID, valuation_date=VALUATION_DATE, export_id="16")

---

## 10. PE Stress Testing

NAV sensitivity scenarios calibrated to the fund strategy, with magnitudes documented in the risk policy (migrated from earlier notebook assumptions, unchanged):

- **S1** uniform NAV markdown on active companies,
- **S2** exit-multiple compression (ΔNAV = ΔMultiple × EBITDA),
- **S3** revenue/EBITDA stress (ΔNAV = ΔEBITDA × Multiple),
- **S4** technology sector concentration shock,
- **S5** 2008 GFC proxy markdown.

Funding-liquidity stress is covered in Section 8; this section focuses on valuation sensitivity.

In [ ]:
ped.display_stress_by_company(workflow["stress_results"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="17")

In [ ]:
ped.display_stress_summary(workflow["stress_results"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="18")

In [ ]:
ped.plot_stress_summary(workflow["stress_results"], FUND_ID, valuation_date=VALUATION_DATE, export_id="19")

---

## 11. Sustainability Risk Indicators

Portfolio-company ESG indicators use the private-asset ESG workflow: scores come from reference data keyed by company, simulating appraiser or fund-administrator assessments, with `esg_reporter` identifying the source. Manager estimates are less reliable than independent assessments and are flagged in the table.

> Scale note: ESG scores use a 0-100 scale, where 100 is best. ESG scores are sustainability-risk inputs and are not mapped to SFDR classifications here.

In [ ]:
esg_u.display_esg_assets(workflow["esg_df"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="20")

In [ ]:
esg_u.display_esg_summary(workflow["esg_df"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="21")

In [ ]:
esg_u.plot_esg_profile(workflow["esg_df"], FUND_ID, plot_title='ESG profile — PE Buyout', valuation_date=VALUATION_DATE, export_id="22")

---

## 12. Annex IV Report

PE buyout funds report under AIFMD Annex IV with fields specific to closed-ended private equity vehicles:

- **Leverage**: only fund-level borrowing (the subscription line) enters the AIFMD leverage calculation; portfolio company debt is ring-fenced at SPV level per the project-finance treatment.
- **Exposures**: reported by sector, country, and investment stage on cost basis — PE positions have no daily market prices.
- **Unfunded commitments**: disclosed as contingent leverage under the AIFMD II expanded disclosures.

**Regulatory basis:** Delegated Regulation (EU) 231/2013 Article 110 and Annex IV reporting template.

In [ ]:
import fund_risk_workflow.reporting.annex_iv_workflow as annex_iv_workflow
from fund_risk_workflow.pipeline.pe_buyout_workflow import PE_SECTIONS

annex_iv_result = annex_iv_workflow.run(
    engine=ENGINE,
    fund_id=FUND_ID,
    quarter=QUARTER,
    first_export_id="23",
    sections=PE_SECTIONS,
)